# GPU Computing on HPC

Tuesday you ran Python on the CPU. Today you'll use the **GPU** — the same hardware that trains AI models and runs physics simulations.

You launched this notebook through the **TACC Analysis Portal** (TAP) instead of SSH tunneling. Same compute node, easier setup.

---
## Part 0: Prove You Have a GPU

In [ ]:
import os
import platform
import time
import numpy as np
%matplotlib inline

print(f"Hostname:  {platform.node()}")
print(f"CPU cores: {os.cpu_count()}")
print(f"Platform:  {platform.machine()}")

with open('/proc/meminfo') as f:
    mem_kb = int(f.readline().split()[1])
    print(f"RAM:       {mem_kb / 1e6:.0f} GB")

print()
gpu_info = os.popen('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null').read().strip()
if gpu_info:
    print(f"GPU:       {gpu_info}")
else:
    print("GPU:       NOT DETECTED -- ask for help!")

You should see an **NVIDIA GH200** with ~98 GB of GPU memory. That GPU has its own processor and its own memory, separate from the CPU.

Think of it this way:
- **CPU** (Grace): 72 cores, great at complex tasks, one at a time
- **GPU** (Hopper): thousands of small cores, great at doing the same simple operation on massive amounts of data at once

---
## Part 1: Hello GPU with CuPy

**CuPy** is NumPy for the GPU. Same syntax, but the math runs on the GPU instead of the CPU.

```
NumPy:  np.array([1, 2, 3])     # lives in CPU memory (RAM)
CuPy:   cp.array([1, 2, 3])     # lives in GPU memory (VRAM)
```

In [ ]:
import cupy as cp

# Check GPU details from CuPy's perspective
gpu_name = os.popen('nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null').read().strip()
device = cp.cuda.Device(0)
mem_free, mem_total = device.mem_info

print(f"GPU device:      {gpu_name}")
print(f"GPU memory:      {mem_total / 1e9:.0f} GB total")
print(f"GPU memory free: {mem_free / 1e9:.0f} GB available")

In [ ]:
# NumPy array -- lives on the CPU
cpu_array = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
print(f"NumPy array: {cpu_array}")
print(f"  Type: {type(cpu_array)}")
print(f"  Lives on: CPU (RAM)")

print()

# CuPy array -- lives on the GPU
gpu_array = cp.array([1.0, 2.0, 3.0, 4.0, 5.0])
print(f"CuPy array:  {gpu_array}")
print(f"  Type: {type(gpu_array)}")
print(f"  Lives on: GPU (VRAM)")

In [ ]:
# Same syntax, different hardware
print("Math on CPU:", cpu_array * 2)
print("Math on GPU:", gpu_array * 2)
print()
print("Dot product on CPU:", np.dot(cpu_array, cpu_array))
print("Dot product on GPU:", cp.dot(gpu_array, gpu_array))

The code looks almost identical. The difference is where the data lives and where the math happens.

**Key idea:** To use the GPU, you move your data to GPU memory. Then all operations on that data run on the GPU automatically.

---
## Part 2: CPU vs GPU — Matrix Multiplication

Tuesday you saw the CPU doing matrix multiplication at about 37 GFLOPS. Let's see what the GPU can do.

We'll time the same operation on both and compare.

In [ ]:
# Warm up the GPU -- the first operation is always slow (loading libraries onto the GPU)
_ = cp.array([1.0]) @ cp.array([1.0])
cp.cuda.Stream.null.synchronize()
print("GPU warmed up.")

In [ ]:
def benchmark_cpu(n):
    """Matrix multiply on CPU using NumPy."""
    A = np.random.randn(n, n).astype(np.float32)
    B = np.random.randn(n, n).astype(np.float32)
    start = time.time()
    C = A @ B
    elapsed = time.time() - start
    return elapsed

def benchmark_gpu(n):
    """Matrix multiply on GPU using CuPy."""
    A = cp.random.randn(n, n, dtype=cp.float32)
    B = cp.random.randn(n, n, dtype=cp.float32)
    cp.cuda.Stream.null.synchronize()  # make sure data is ready
    start = time.time()
    C = A @ B
    cp.cuda.Stream.null.synchronize()  # wait for GPU to finish
    elapsed = time.time() - start
    return elapsed

In [ ]:
sizes = [500, 1000, 2000, 5000, 10000]
cpu_times = []
gpu_times = []

print(f"{'Size':>8} {'CPU (sec)':>12} {'GPU (sec)':>12} {'Speedup':>10} {'GPU TFLOPS':>12}")
print("-" * 58)

for n in sizes:
    t_cpu = benchmark_cpu(n)
    t_gpu = benchmark_gpu(n)
    
    cpu_times.append(t_cpu)
    gpu_times.append(t_gpu)
    
    speedup = t_cpu / t_gpu
    flops = 2 * n**3
    gpu_tflops = flops / t_gpu / 1e12
    
    print(f"{n:>8} {t_cpu:>12.4f} {t_gpu:>12.4f} {speedup:>9.1f}x {gpu_tflops:>11.2f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Time comparison
x = range(len(sizes))
width = 0.35
axes[0].bar([i - width/2 for i in x], cpu_times, width, label='CPU (NumPy)', color='steelblue')
axes[0].bar([i + width/2 for i in x], gpu_times, width, label='GPU (CuPy)', color='maroon')
axes[0].set_xticks(x)
axes[0].set_xticklabels([str(s) for s in sizes])
axes[0].set_xlabel('Matrix Size', fontsize=12)
axes[0].set_ylabel('Time (seconds)', fontsize=12)
axes[0].set_title('CPU vs GPU Time', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Speedup
speedups = [c / g for c, g in zip(cpu_times, gpu_times)]
axes[1].bar(x, speedups, color='maroon')
axes[1].set_xticks(x)
axes[1].set_xticklabels([str(s) for s in sizes])
axes[1].set_xlabel('Matrix Size', fontsize=12)
axes[1].set_ylabel('Speedup (x times faster)', fontsize=12)
axes[1].set_title('GPU Speedup over CPU', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### YOUR TURN

1. At what matrix size does the GPU start clearly beating the CPU?
2. Why is the GPU speedup smaller for the 500x500 case than for the 10,000x10,000 case?

---
## Part 3: The Data Transfer Cost

The GPU is fast at math, but there's a catch: data has to travel between CPU memory (RAM) and GPU memory (VRAM). That transfer takes time.

```
CPU Memory (RAM)  ──transfer──>  GPU Memory (VRAM)
                                    │
                                 GPU does math
                                    │
CPU Memory (RAM)  <──transfer──  GPU Memory (VRAM)
```

For small problems, the transfer time can be longer than the actual computation.

In [ ]:
# Measure: how long does it take to move data to the GPU?
sizes_mb = [1, 10, 100, 500, 1000]

print(f"{'Data Size':>12} {'Transfer Time':>15} {'Bandwidth':>15}")
print("-" * 45)

for mb in sizes_mb:
    n_floats = mb * 1_000_000 // 4  # float32 = 4 bytes
    cpu_data = np.random.randn(n_floats).astype(np.float32)
    
    cp.cuda.Stream.null.synchronize()
    start = time.time()
    gpu_data = cp.asarray(cpu_data)        # CPU -> GPU
    cp.cuda.Stream.null.synchronize()
    elapsed = time.time() - start
    
    bandwidth = mb / elapsed / 1000  # GB/s
    print(f"{mb:>8} MB {elapsed:>12.4f} s {bandwidth:>12.1f} GB/s")
    
    del gpu_data

print()
print("The GH200 has unified memory, so transfers are fast.")
print("On a traditional GPU (PCIe), this would be much slower.")

### YOUR TURN

1. If you have a 100 MB dataset and a computation that takes 0.001 seconds on the GPU, is it worth moving the data to the GPU? Why or why not?
2. The GH200 connects CPU and GPU with a special high-speed link (NVLink-C2C) instead of PCIe. Why does that matter?

---
## Part 4: Real-World GPU Use — Image Processing at Scale

GPUs aren't just for matrix math. Any operation you need to repeat millions of times in parallel is a GPU problem.

Here's a common example: applying a filter to an image. We'll create a large synthetic image and blur it on both CPU and GPU.

In [ ]:
from cupyx.scipy.ndimage import uniform_filter as gpu_filter
from scipy.ndimage import uniform_filter as cpu_filter

# Create a large "image" (8K resolution, grayscale)
height, width = 8000, 8000
image_cpu = np.random.rand(height, width).astype(np.float32)
image_gpu = cp.asarray(image_cpu)

print(f"Image size: {height} x {width} = {height * width / 1e6:.0f} million pixels")
print(f"Memory: {image_cpu.nbytes / 1e6:.0f} MB")
print()

# CPU blur
start = time.time()
blurred_cpu = cpu_filter(image_cpu, size=15)
t_cpu = time.time() - start
print(f"CPU blur: {t_cpu:.3f} seconds")

# GPU blur
cp.cuda.Stream.null.synchronize()
start = time.time()
blurred_gpu = gpu_filter(image_gpu, size=15)
cp.cuda.Stream.null.synchronize()
t_gpu = time.time() - start
print(f"GPU blur: {t_gpu:.3f} seconds")

print(f"\nSpeedup: {t_cpu / t_gpu:.1f}x")

In [ ]:
# Visualize a small section
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Show a 500x500 crop for visibility
crop = slice(2000, 2500)
axes[0].imshow(image_cpu[crop, crop], cmap='gray')
axes[0].set_title('Original (500x500 crop)', fontsize=14)
axes[0].axis('off')

axes[1].imshow(cp.asnumpy(blurred_gpu)[crop, crop], cmap='gray')
axes[1].set_title('GPU-Blurred', fontsize=14)
axes[1].axis('off')

plt.tight_layout()
plt.show()

---
## Part 5: The Big Picture — When to Use What

Now you've seen three ways to compute: single CPU core, multiple CPU cores, and GPU.

Let's put it all together with one final comparison.

In [ ]:
# The 20,000x20,000 matrix multiply from Tuesday took ~430 seconds on the CPU.
# Let's do it on the GPU.

n = 20000

print(f"Matrix size: {n:,} x {n:,}")
print(f"Operations:  {2 * n**3 / 1e12:.1f} trillion")
print()

# GPU version
A_gpu = cp.random.randn(n, n, dtype=cp.float32)
B_gpu = cp.random.randn(n, n, dtype=cp.float32)
cp.cuda.Stream.null.synchronize()

start = time.time()
C_gpu = A_gpu @ B_gpu
cp.cuda.Stream.null.synchronize()
t_gpu = time.time() - start

gpu_tflops = 2 * n**3 / t_gpu / 1e12

print(f"GPU time:       {t_gpu:.1f} seconds")
print(f"GPU throughput: {gpu_tflops:.1f} TFLOPS")
print()
print(f"Tuesday's CPU time: ~430 seconds")
print(f"Today's GPU time:   {t_gpu:.1f} seconds")
print(f"Speedup:            ~{430 / t_gpu:.0f}x")

del A_gpu, B_gpu, C_gpu

### Summary: When to Use What

| Approach | Good For | Example |
|----------|----------|---------|
| **1 CPU core** | Small data, complex logic, quick scripts | Reading a CSV, string processing |
| **Many CPU cores** | Independent tasks that can run in parallel | Monte Carlo simulations, file processing |
| **GPU** | Same operation on millions of data points | Matrix math, image processing, AI training |

**Rule of thumb:** If your problem is big and repetitive, try the GPU. If it's complex and branching, stick with the CPU.

---
## Exit Ticket

Submit these answers on Canvas.

**1. What is the name of the GPU on your compute node, and how much GPU memory does it have?**

_Your answer:_


**2. How much faster was the GPU than the CPU for the 10,000x10,000 matrix multiply?**

_Your answer:_


**3. What does CuPy do? How is it different from NumPy?**

_Your answer:_


**4. Name one scenario where a GPU would NOT be faster than a CPU.**

_Your answer:_